# Rakshak AI — Offline Medical Triage Fine-Tuning

Fine-tuning Gemma 4 E2B IT on Indian medical datasets for offline disaster triage.

**Data Sources (auto-detected)**:
- HiMed Hindi Medical Corpus (Kaggle, if uploaded)
- Public HF medical QA datasets (auto-fallback)
- Massive synthetic India disaster triage scenarios (10,000+)

**Output**: LoRA adapter → `.litertlm` for on-device inference via flutter_gemma.

In [ ]:
# Cell 1: Install & imports
print("=== Rakshak AI Fine-Tuning ===")
print("Installing dependencies...")

import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Set CUDA arch for T4 GPU (sm_75) compatibility
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Install Unsloth — Kaggle T4x2 compatible
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

import torch
import json
import re
import zipfile
import random
import numpy as np
from datasets import Dataset, load_dataset, concatenate_datasets
from unsloth import FastModel, is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments
from transformers import TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

print("\n=== GPU Check ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} GB")

print("\n=== Environment ===")
print(f"Kaggle working dir: {os.getcwd()}")
print(f"Files in /kaggle/input/: {os.listdir('/kaggle/input/') if os.path.exists('/kaggle/input/') else 'N/A'}")

---
# 1. Load Datasets
---

In [ ]:
# Cell 2: Dataset Discovery — search Kaggle input paths for a dataset
print("=" * 60)
print("Dataset Discovery")
print("=" * 60)

def find_kaggle_dataset(possible_names, required_files=None):
    """Search Kaggle input paths for a dataset (depth-first, skips giant system dirs)."""
    base = "/kaggle/input"
    if not os.path.exists(base):
        return None
    # Collect all directories up to 3 levels deep, skipping known-large system dirs
    SKIP = {"competitions", "models", "kernels", "sources", "hub", "config"}
    candidates = []
    for entry in sorted(os.listdir(base), reverse=True):
        ep = os.path.join(base, entry)
        if not os.path.isdir(ep) or entry in SKIP:
            continue
        candidates.append(ep)
        # Level 2
        try:
            for sub1 in sorted(os.listdir(ep), reverse=True):
                sp1 = os.path.join(ep, sub1)
                if os.path.isdir(sp1):
                    candidates.append(sp1)
                    # Level 3
                    try:
                        for sub2 in sorted(os.listdir(sp1), reverse=True):
                            sp2 = os.path.join(sp1, sub2)
                            if os.path.isdir(sp2):
                                candidates.append(sp2)
                    except PermissionError:
                        pass
        except PermissionError:
            pass
    for cpath in candidates:
        dirname = os.path.basename(cpath)
        for name in possible_names:
            if name.lower() in dirname.lower():
                if required_files:
                    for root, dirs, files in os.walk(cpath):
                        if any(f.endswith(ext) for f in files for ext in required_files):
                            return cpath
                else:
                    return cpath
    return None

found_datasets = {}


### HiMed Hindi Medical Dataset (Kaggle — optional)

In [ ]:
# Cell 3: Load HiMed if available
print("=" * 60)
print("Loading HiMed Hindi Medical Dataset (Kaggle)...")
print("=" * 60)

himed_data = []
himed_path = find_kaggle_dataset(
    ["himed", "hindi-medical"],
    required_files=[".json", ".jsonl", ".csv"]
)

if himed_path:
    print(f"Found HiMed at: {himed_path}")
    all_files = []
    for root, dirs, files in os.walk(himed_path):
        for f in files:
            if f.endswith('.json') or f.endswith('.jsonl') or f.endswith('.csv'):
                all_files.append(os.path.join(root, f))
    print(f"Found {len(all_files)} data files")
    for fpath in sorted(all_files):
        try:
            with open(fpath, 'r', encoding='utf-8') as f:
                if fpath.endswith('.jsonl'):
                    for line in f:
                        himed_data.append(json.loads(line))
                elif fpath.endswith('.json'):
                    data = json.load(f)
                    if isinstance(data, list):
                        himed_data.extend(data)
                    elif isinstance(data, dict):
                        himed_data.append(data)
                elif fpath.endswith('.csv'):
                    import csv
                    reader = csv.DictReader(f)
                    for row in reader:
                        himed_data.append(row)
            print(f"  Loaded {fpath}: {len(himed_data)} total")
        except Exception as e:
            print(f"  Skipped {fpath}: {e}")
    found_datasets['himed'] = len(himed_data)
else:
    print("HiMed not found on Kaggle.")
    print("To use HiMed: Upload the dataset to Kaggle with 'himed' in the name.")

print(f"\nTotal HiMed samples: {len(himed_data)}")

### Public Medical QA Datasets (HuggingFace — auto-fallback)

In [ ]:
public_medical_data = []

# Dataset 1: medmcqa (built-in datasets library)
try:
    print("\nTrying medmcqa...")
    medmcqa_ds = load_dataset("medmcqa", split="train")
    count = 0
    for item in medmcqa_ds:
        if count >= 3000:
            break
        q = item.get('question', '')
        opts = (item.get('opa',''), item.get('opb',''), item.get('opc',''), item.get('opd',''))
        ans_idx = item.get('cop', 0)
        if q and any(opts):
            choices = ['A','B','C','D']
            correct = choices[ans_idx] if ans_idx < 4 else 'A'
            public_medical_data.append({
                "instruction": f"Medical question: {q}",
                "input": "\n".join([f"{c}: {o}" for c,o in zip(choices,opts) if o]),
                "output": f"The correct answer is {correct}."
            })
            count += 1
    print(f"  Loaded {count} medmcqa samples")
except Exception as e:
    print(f"  medmcqa unavailable: {e}")

# Dataset 2: pubmed_qa (built-in datasets library)
try:
    print("\nTrying pubmed_qa...")
    pubmed_ds = load_dataset("pubmed_qa", "pqa_labeled", split="train")
    count = 0
    for item in pubmed_ds:
        if count >= 3000:
            break
        q = item.get('question', '')
        ctx = item.get('context', '')
        a = item.get('long_answer', '')
        if q and a:
            public_medical_data.append({
                "instruction": f"Medical Question: {q}",
                "input": f"Context: {ctx[:500]}" if ctx else "",
                "output": a
            })
            count += 1
    print(f"  Loaded {count} pubmed_qa samples")
except Exception as e:
    print(f"  pubmed_qa unavailable: {e}")

# Dataset 3: GBaker/MedQA-USMLE-4-options (Parquet format)
try:
    print("\nTrying GBaker/MedQA-USMLE...")
    usmle_ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")
    count = 0
    for item in usmle_ds:
        if count >= 2000:
            break
        q = item.get('question', '')
        opts = [item[k] for k in ['op0','op1','op2','op3'] if item.get(k)]
        ans = item.get('answer', '')
        if q and ans:
            public_medical_data.append({
                "instruction": f"Medical question: {q}",
                "input": "\n".join([f"{i+1}. {o}" for i,o in enumerate(opts)]) if opts else "",
                "output": f"Answer: {ans}"
            })
            count += 1
    print(f"  Loaded {count} USMLE samples")
except Exception as e:
    print(f"  GBaker/MedQA-USMLE unavailable: {e}")

print(f"\nTotal public medical QA samples: {len(public_medical_data)}")
if len(public_medical_data) > 0:
    print(f"  Example: {public_medical_data[0]['instruction'][:100]}...")

---
# 2. Convert Datasets to Training Format
---

In [ ]:
# Cell 6: Convert HiMed to instruction format
print("=" * 60)
print("Converting HiMed to training format...")
print("=" * 60)

def convert_himed_to_chat(entry):
    if isinstance(entry, dict):
        instruction = entry.get('instruction', '')
        inp = entry.get('input', '')
        output = entry.get('output', '')
        text = entry.get('text', '')
        summary = entry.get('summary', '')
        
        if instruction and output:
            user_text = instruction
            if inp:
                user_text += f"\n\n{inp}"
            return {
                "instruction": instruction,
                "input": inp,
                "output": output
            }
        elif text and summary:
            return {
                "instruction": text[:500],
                "input": "",
                "output": summary
            }
    return None

himed_chat = []
for entry in himed_data:
    converted = convert_himed_to_chat(entry)
    if converted:
        himed_chat.append(converted)

print(f"Converted {len(himed_chat)} HiMed samples")
if len(himed_chat) > 0:
    print(f"  Example output: {himed_chat[0]}")

In [ ]:
# Cell 7: Convert public medical QA to training format
print("=" * 60)
print("Converting public medical QA to training format...")
print("=" * 60)

def convert_medical_qa(item):
    """Convert a medical QA item to our instruction format."""
    if not isinstance(item, dict):
        return None
    instruction = item.get('instruction', '')
    inp = item.get('input', '')
    output = item.get('output', '')
    if instruction and output:
        return {
            "instruction": instruction,
            "input": inp or '',
            "output": output
        }
    return None

public_medical_chat = []
for entry in public_medical_data:
    converted = convert_medical_qa(entry)
    if converted:
        public_medical_chat.append(converted)

print(f"Converted {len(public_medical_chat)} public medical QA samples")
if len(public_medical_chat) > 0:
    print(f"  Example: {public_medical_chat[0]['instruction'][:100]}...")

---
# 3. Generate Synthetic India Disaster Triage Data (10,000+)
---

In [ ]:
# Cell 8: Synthetic India disaster triage generator — expanded
print("=" * 60)
print("Generating Synthetic India Disaster Triage Data (10,000+)...")
print("=" * 60)

random.seed(42)
np.random.seed(42)

# ── Disaster scenarios ──
disasters = [
    ("flood", ["बिहार", "असम", "उत्तर प्रदेश", "पश्चिम बंगाल", "गुजरात", "केरल", "ओडिशा"]),
    ("cyclone", ["ओडिशा", "आंध्र प्रदेश", "तमिलनाडु", "पश्चिम बंगाल"]),
    ("earthquake", ["गुजरात", "हिमाचल प्रदेश", "उत्तराखंड", "जम्मू और कश्मीर"]),
    ("heatwave", ["राजस्थान", "मध्य प्रदेश", "उत्तर प्रदेश", "दिल्ली"]),
    ("landslide", ["हिमाचल प्रदेश", "उत्तराखंड", "केरल", "पश्चिम बंगाल"]),
    ("fire", ["मुंबई", "दिल्ली", "कोलकाता", "चेन्नई", "बैंगलोर"]),
    ("riot", ["दिल्ली", "मुंबई", "लखनऊ", "अहमदाबाद"]),
    ("building_collapse", ["मुंबई", "दिल्ली", "कोलकाता", "चेन्नई"]),
    ("chemical_spill", ["गुजरात", "महाराष्ट्र", "तमिलनाडु", "उत्तर प्रदेश"]),
    ("drought", ["महाराष्ट्र", "कर्नाटक", "राजस्थान", "मध्य प्रदेश"]),
    ("thunderstorm", ["पश्चिम बंगाल", "झारखंड", "बिहार", "असम"]),
    ("epidemic", ["केरल", "महाराष्ट्र", "दिल्ली", "गुजरात"]),
]

disaster_hindi_map = {
    "flood": "बाढ़", "cyclone": "चक्रवात", "earthquake": "भूकंप",
    "heatwave": "भीषण गर्मी", "landslide": "भूस्खलन", "fire": "आग",
    "riot": "दंगा", "building_collapse": "भवन ढहना",
    "chemical_spill": "रासायनिक रिसाव", "drought": "सूखा",
    "thunderstorm": "तूफान", "epidemic": "महामारी",
}

# ── 40+ injury patterns with proper START mapping ──
injuries = [
    # RED — immediate (life-threatening)
    ("सिर में गंभीर चोट", "severe head injury", "RED", "airway/breathing compromise from head trauma"),
    ("बेहोश", "unconscious", "RED", "unconscious patient with potential airway obstruction"),
    ("भारी रक्तस्राव", "heavy uncontrollable bleeding", "RED", "active hemorrhage with signs of shock"),
    ("सांस नहीं ले पा रहा", "not breathing", "RED", "respiratory arrest requiring immediate intervention"),
    ("पानी में डूब रहा", "drowning", "RED", "respiratory emergency from water aspiration"),
    ("मलबे में दबा", "trapped under debris", "RED", "crush injury with potential airway compromise"),
    ("हीट स्ट्रोक", "heat stroke", "RED", "hyperthermia with altered mental status"),
    ("बिजली का झटका", "electric shock", "RED", "cardiac rhythm disturbance from electrocution"),
    ("छुरा घाव छाती में", "chest stab wound", "RED", "penetrating chest trauma with breathing difficulty"),
    ("गर्भवती महिला को संकुचन", "pregnant woman in active labor", "RED", "obstetric emergency requiring immediate care"),
    ("सीने में दर्द और सांस फूलना", "chest pain and shortness of breath", "RED", "possible heart attack — requires immediate attention"),
    ("जहरीली गैस का असर", "toxic gas inhalation", "RED", "respiratory distress from chemical exposure"),
    ("रीढ़ की हड्डी में चोट", "spinal cord injury", "RED", "potential spinal damage with neurological deficit"),
    ("गंभीर जलन >40% शरीर", "severe burns >40% BSA", "RED", "extensive burns with airway risk"),
    ("कई अंगों में गंभीर चोट", "multiple severe injuries", "RED", "polytrauma with unstable vital signs"),
    # YELLOW — delayed (serious but stable)
    ("हड्डी टूट गई", "broken bone fracture", "YELLOW", "closed fracture, neurovascular status intact"),
    ("जलने का घाव", "burn injury partial thickness", "YELLOW", "moderate burns, airway clear"),
    ("तेज बुखार", "high fever >102F", "YELLOW", "febrile illness, hemodynamically stable"),
    ("उल्टी और दस्त", "vomiting and diarrhea", "YELLOW", "GI distress with dehydration risk"),
    ("डिहाइड्रेशन", "dehydration", "YELLOW", "mild to moderate dehydration, stable vitals"),
    ("धुएं से दम घुटना", "smoke inhalation", "YELLOW", "mild respiratory irritation, alert"),
    ("गहरा कट जिसमें टांके चाहिए", "deep laceration needing sutures", "YELLOW", "wound requiring closure, bleeding controlled"),
    ("पेट में तेज दर्द", "severe abdominal pain", "YELLOW", "abdominal pain, peritonitis signs absent"),
    ("बच्चे को तेज बुखार और दौरे", "child with febrile seizures", "YELLOW", "pediatric fever with seizures, now stable"),
    ("बुजुर्ग व्यक्ति गिर गया और हिल नहीं पा रहा", "elderly fall with immobility", "YELLOW", "fall with possible fracture, stable vitals"),
    ("आंख में रासायनिक पदार्थ", "chemical splash in eye", "YELLOW", "eye irritation, vision not affected"),
    ("मामूली सिर चोट", "minor head injury", "YELLOW", "head trauma without LOC, alert"),
    ("साँप का काटना", "snake bite", "YELLOW", "snake envenomation, no respiratory compromise yet"),
    ("एलर्जी प्रतिक्रिया", "moderate allergic reaction", "YELLOW", "urticaria, no airway compromise"),
    ("कान या नाक से खून", "ear or nose bleeding", "YELLOW", "ENT bleeding, hemodynamically stable"),
    # GREEN — minimal (walking wounded)
    ("हल्की चोट", "minor injury abrasion", "GREEN", "superficial wound, no functional limitation"),
    ("खरोंच और निशान", "scratches and bruises", "GREEN", "minor soft tissue injury, ambulatory"),
    ("घबराहट और डर", "anxiety and fear", "GREEN", "psychological distress, physically unharmed"),
    ("मोच", "sprain ankle", "GREEN", "mild ligament injury, able to walk"),
    ("सिरदर्द और चक्कर", "headache and dizziness", "GREEN", "mild symptoms, no neurological deficit"),
    ("हल्का बुखार", "mild fever", "GREEN", "low-grade fever, fully mobile"),
    ("थकान और कमजोरी", "fatigue and weakness", "GREEN", "exhaustion, needs rest but no immediate danger"),
    ("मामूली जलन", "minor burn superficial", "GREEN", "first-degree burn, small area"),
    ("भूख और प्यास", "hunger and thirst", "GREEN", "basic needs, no medical emergency"),
    ("दवा की जरूरत", "needs medication refill", "GREEN", "chronic medication requirement, stable"),
]

# ── Victim profiles with age, gender, descriptors ──
victim_profiles = [
    ("5 वर्ष का बच्चा", "5-year-old child", "child"),
    ("8 वर्ष की बच्ची", "8-year-old girl", "child"),
    ("25 वर्ष का युवक", "25-year-old young man", "adult"),
    ("30 वर्ष की महिला", "30-year-old woman", "adult"),
    ("40 वर्ष का पुरुष", "40-year-old man", "adult"),
    ("18 वर्ष की युवती", "18-year-old young woman", "adult"),
    ("60 वर्ष के बुजुर्ग", "60-year-old elderly man", "elderly"),
    ("70 वर्ष की वृद्ध महिला", "70-year-old elderly woman", "elderly"),
    ("35 वर्ष की गर्भवती महिला", "35-year-old pregnant woman", "pregnant"),
    ("2 वर्ष का शिशु", "2-year-old infant", "infant"),
    ("45 वर्ष का व्यक्ति", "45-year-old person", "adult"),
    ("55 वर्ष की महिला", "55-year-old woman", "adult"),
]

# ── Clinical status variations ──
statuses = [
    "सचेत है और बोल सकता है", "conscious and talking",
    "अर्ध-सचेत है", "semi-conscious",
    "बेहोश है", "unconscious",
    "भ्रमित और डरा हुआ है", "confused and frightened",
    "दर्द से कराह रहा है", "moaning in pain",
    "शांत लेकिन पीला दिख रहा है", "quiet but looks pale",
    "बहुत बेचैन है", "very agitated",
]

respiratory = [
    "सामान्य सांस ले रहा है", "breathing normally",
    "तेज सांस ले रहा है (>30 प्रति मिनट)", "rapid breathing >30/min",
    "सांस बहुत धीमी है (<10 प्रति मिनट)", "very slow breathing <10/min",
    "सांस नहीं ले पा रहा", "not breathing",
    "घरघराहट के साथ सांस ले रहा है", "wheezing",
    "सांस लेने में तकलीफ", "difficulty breathing",
]

def generate_scenario(lang='hindi'):
    disaster_eng, regions = random.choice(disasters)
    region = random.choice(regions)
    injury_hindi, injury_eng, triage, clinical_reason = random.choice(injuries)
    v_hindi, v_eng, v_age_group = random.choice(victim_profiles)
    
    # Pick appropriate status based on severity
    if triage == 'RED':
        valid = [s for s in statuses if statuses.index(s) % 2 == 0 and s not in ["सचेत है और बोल सकता है", "शांत लेकिन पीला दिख रहा है"]]
        # Get even-indexed (hindi) strings only
        valid_hindi = [statuses[i] for i in range(0, len(statuses), 2) if statuses[i] not in ["सचेत है और बोल सकता है", "शांत लेकिन पीला दिख रहा है"]]
        status_hindi = random.choice(valid_hindi)
        status_eng = statuses[statuses.index(status_hindi) + 1]
    elif triage == 'GREEN':
        status_hindi = "सचेत है और बोल सकता है"
        status_eng = "conscious and talking"
    else:
        status_hindi = random.choice([statuses[i] for i in range(0, len(statuses), 2)])
        status_eng = statuses[statuses.index(status_hindi) + 1]
    
    # Respiratory pattern
    resp_hindi = random.choice([respiratory[i] for i in range(0, len(respiratory), 2)])
    resp_eng = respiratory[respiratory.index(resp_hindi) + 1]
    
    # Capillary refill / circulation
    if triage == 'RED':
        cap_refill = random.choice([">2 सेकंड", ">2 seconds", "कमज़ोर नाड़ी", "weak pulse"])
    elif triage == 'YELLOW':
        cap_refill = random.choice(["<2 सेकंड", "<2 seconds", "सामान्य", "normal"])
    else:
        cap_refill = random.choice(["<2 सेकंड", "<2 seconds", "सामान्य", "normal"])
    
    # Location
    location_templates = [
        f"{region} के एक गाँव में", f"in a village in {region}",
        f"{region} के बाहरी इलाके में", f"on the outskirts of {region}",
        f"{region} के राहत शिविर में", f"at a relief camp in {region}",
        f"{region} के शहरी क्षेत्र में", f"in an urban area of {region}",
        f"{region} की सड़क पर", f"on a road in {region}",
        f"{region} के अस्पताल के बाहर", f"outside a hospital in {region}",
    ]
    loc_pair = random.choice(location_templates)
    # Extract by parity
    all_locs = location_templates
    idx = all_locs.index(loc_pair)
    if idx % 2 == 0:
        loc_hindi, loc_eng = all_locs[idx], all_locs[idx + 1]
    else:
        loc_hindi, loc_eng = all_locs[idx - 1], all_locs[idx]
    
    disaster_hindi = disaster_hindi_map.get(disaster_eng, disaster_eng)
    
    if lang == 'hindi':
        user = (
            f"एक आपातकालीन स्थिति: {loc_hindi} {disaster_hindi} आई है। "
            f"पीड़ित: {v_hindi} जिसे {injury_hindi} लगी है। "
            f"स्थिति: {status_hindi}। "
            f"सांस: {resp_hindi}। "
            f"परिसंचरण: {cap_refill}। "
            f"कृपया START ट्रायेज प्रोटोकॉल के अनुसार श्रेणी बताएं।"
        )
        reasons = {
            "RED": "जानलेवा स्थिति — तत्काल उपचार की आवश्यकता है",
            "YELLOW": "गंभीर लेकिन स्थिर — 1-2 घंटे में उपचार संभव",
            "GREEN": "हल्की चोट — चलने-फिरने में सक्षम, उपचार में देरी हो सकती है",
        }
        response = (
            f"START ट्रायेज श्रेणी: {triage}\n\n"
            f"कारण: {reasons[triage]}। "
            f"पीड़ित को {injury_hindi} ({injury_eng}) है, "
            f"{resp_hindi.lower()}। "
            f"{clinical_reason}।"
        )
    else:
        user = (
            f"Emergency situation: {loc_eng} after a {disaster_eng}. "
            f"Victim: {v_eng} with {injury_eng}. "
            f"Status: {status_eng}. "
            f"Breathing: {resp_eng}. "
            f"Circulation: {cap_refill}. "
            f"What is the START triage category?"
        )
        reasons = {
            "RED": "Life-threatening condition — immediate treatment required",
            "YELLOW": "Serious but stable — can wait 1-2 hours for treatment",
            "GREEN": "Minor injury — ambulatory, treatment can be delayed",
        }
        response = (
            f"START Triage Category: {triage}\n\n"
            f"Reason: {reasons[triage]}. "
            f"The victim has {injury_eng}. "
            f"{clinical_reason}."
        )
    
    return {
        "instruction": user,
        "input": "",
        "output": response,
    }

# Generate 8000 Hindi + 2000 English = 10000 total
synthetic_triage = [generate_scenario('hindi') for _ in range(8000)]
synthetic_triage += [generate_scenario('english') for _ in range(2000)]
random.shuffle(synthetic_triage)

print(f"Generated {len(synthetic_triage)} synthetic India disaster triage scenarios")
print(f"  Hindi: 8000, English: 2000 (shuffled)")

# Check category distribution
from collections import Counter
cats = []
for s in synthetic_triage:
    for c in ['RED', 'YELLOW', 'GREEN']:
        if c in s['output']:
            cats.append(c)
            break
print(f"Triage balance: {Counter(cats)}")

if len(synthetic_triage) > 0:
    print(f"\nExample Hindi scenario:")
    print(f"  User: {synthetic_triage[0]['instruction'][:200]}...")
    print(f"  Output: {synthetic_triage[0]['output'][:200]}...")

---
# 4. Combine All Datasets
---

In [ ]:
# Cell 10: Combine all datasets
print("=" * 60)
print("Combining All Datasets...")
print("=" * 60)

# To prevent over-representation, cap larger datasets
himed_capped = random.sample(himed_chat, min(5000, len(himed_chat))) if himed_chat else []

# Public medical QA capped at 3000
public_capped = random.sample(public_medical_chat, min(3000, len(public_medical_chat))) if public_medical_chat else []

# Use all synthetic (10000)
synthetic_all = synthetic_triage

# Combine
all_data = himed_capped + public_capped + synthetic_all
random.shuffle(all_data)

print(f"Dataset composition:")
print(f"  HiMed (Hindi Medical): {len(himed_capped)} samples")
print(f"  Public Medical QA: {len(public_capped)} samples")
print(f"  Synthetic India Disaster: {len(synthetic_all)} samples")
print(f"  TOTAL: {len(all_data)} samples")

# Save combined dataset
os.makedirs("/kaggle/working/rakshak-data", exist_ok=True)
with open("/kaggle/working/rakshak-data/combined_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in all_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print(f"Saved combined dataset to /kaggle/working/rakshak-data/combined_dataset.jsonl")

---
# 5. Load Gemma 4 E2B & Apply LoRA
---

In [ ]:
# Cell 11: Load Gemma 4 E2B IT with Unsloth
print("=" * 60)
print("Loading Gemma 4 E2B IT...")
print("=" * 60)

model_id = "unsloth/gemma-4-e2b-it-unsloth-bnb-4bit"

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    device_map={"": torch.cuda.current_device()}
)

print(f"Model loaded: {model_id}")
print(f"Max seq length: {max_seq_length}")
print(f"4-bit: {load_in_4bit}")

# Apply LoRA configuration
model = FastModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA applied successfully!")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

---
# 6. Prepare Chat Format & Train
---

In [ ]:
# Cell 12: Load dataset & format with chat template
print("=" * 60)
print("Preparing Training Dataset...")
print("=" * 60)

with open("/kaggle/working/rakshak-data/combined_dataset.jsonl", "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

print(f"Loaded {len(raw_data)} training examples")

def format_chat_template(example):
    """Format using Gemma-4 chat template"""
    instruction = example["instruction"]
    inp = example.get("input", "")
    output = example["output"]
    
    if inp:
        user_content = f"{instruction}\n\n{inp}"
    else:
        user_content = instruction
    
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    
    return {"text": text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat_template, remove_columns=dataset.column_names)

# Split into train (90%) and validation (10%)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")
print(f"\nExample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])

In [ ]:
# Cell 13: Train the model
print("=" * 60)
print("Starting TRAINING...")
print("=" * 60)

# Calculate effective batch size for available GPUs
num_gpus = torch.cuda.device_count()
per_device_batch = 2
grad_accum = 4
effective_batch = per_device_batch * grad_accum * max(1, num_gpus)
total_steps = len(train_dataset) // effective_batch

print(f"GPUs: {num_gpus}")
print(f"Per-device batch: {per_device_batch}")
print(f"Gradient accumulation: {grad_accum}")
print(f"Effective batch size: {effective_batch}")
print(f"Total training steps (approx): {total_steps}")
print(f"Training samples: {len(train_dataset)}")

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=UnslothTrainingArguments(
        per_device_train_batch_size=per_device_batch,
        gradient_accumulation_steps=grad_accum,
        warmup_steps=30,
        num_train_epochs=1,
        learning_rate=2e-4,
        embedding_learning_rate=1e-5,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="/kaggle/working/rakshak-lora",
        report_to="none",
        save_strategy="epoch",
        eval_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=2,
        metric_for_best_model="eval_loss",
        ddp_find_unused_parameters=False if num_gpus > 1 else None,
    ),
)

print("\nTraining...")
trainer.train()

In [ ]:
# Cell 14: Save LoRA adapter
print("=" * 60)
print("Saving LoRA Adapter...")
print("=" * 60)

lora_path = "/kaggle/working/rakshak-lora-final"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"LoRA adapter saved to: {lora_path}")

---
# 7. Inference Test (Before Merge)
---

In [ ]:
# Cell 15: Quick inference test
print("=" * 60)
print("Inference Test...")
print("=" * 60)

FastModel.for_inference(model)

test_cases = [
    "बाढ़ प्रभावित क्षेत्र में एक 30 वर्षीय महिला बेहोश मिली। सांस नहीं ले पा रही। ट्रायेज श्रेणी क्या होगी?",
    "25 वर्षीय व्यक्ति के पैर में मोच आ गई है। वह चल सकता है और सचेत है। ट्रायेज श्रेणी बताएं।",
    "A 60-year-old man in Delhi heatwave has high fever and vomiting. What is the triage category?"
]

for i, test_input in enumerate(test_cases):
    print(f"\n--- Test {i+1} ---")
    print(f"User: {test_input}")
    
    messages = [
        {"role": "user", "content": [{"type": "text", "text": test_input}]}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Assistant: {response}")

---
# 8. Merge LoRA → Base & Export to GGUF/TFLite
---

In [ ]:
# Cell 16: Save merged model (16-bit) for TFLite conversion
print("=" * 60)
print("Saving Merged Model (16-bit)...")
print("=" * 60)

merged_path = "/kaggle/working/rakshak-merged-16bit"

# Save with merge
model.save_pretrained_merged(
    merged_path,
    tokenizer,
    save_method="merged_16bit"
)

print(f"Merged model saved to: {merged_path}")
print(f"Contents: {os.listdir(merged_path)[:10]}...")

In [ ]:
# Cell 17: Export to .litertlm for flutter_gemma
print("=" * 60)
print("Exporting to .litertlm format...")
print("=" * 60)

# Attempt ai-edge-torch conversion; graceful fallback if not available
litertlm_path = "/kaggle/working/rakshak-ai.litertlm"
try:
    !pip install -q ai-edge-torch
    import ai_edge_torch
    from ai_edge_torch.generative.utilities import export as ai_edge_export
    
    # Load merged model for export
    merged_path = "/kaggle/working/rakshak-merged-16bit"
    
    # Export to .litertlm format
    ai_edge_export.model_to_litert(
        merged_path,
        output_path=litertlm_path,
        tokenizer=tokenizer,
        seq_length=2048,
    )
    print(f"Exported to {litertlm_path}")
    print(f"Size: {os.path.getsize(litertlm_path) / 1024**3:.2f} GB")
except ImportError:
    print("ai-edge-torch not available in this Kaggle environment.")
    print("The merged SafeTensors model is saved for offline conversion.")
    print(f"\nTo convert on a local machine:")
    print(f"  1. Download /kaggle/working/rakshak-merged-16bit/")
    print(f"  2. pip install ai-edge-torch")
    print(f"  3. Run conversion script: convert_to_litertlm.py")

In [ ]:
# Cell 18: Package LoRA adapter for Flutter
print("=" * 60)
print("Packaging deliverables...")
print("=" * 60)

# Create zip with LoRA adapter
with zipfile.ZipFile(
    "/kaggle/working/rakshak-ai-lora.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for root, dirs, files in os.walk(lora_path):
        for fn in files:
            file_path = os.path.join(root, fn)
            arcname = os.path.relpath(file_path, lora_path)
            zf.write(file_path, arcname)
            print(f"  Added: {arcname}")

print(f"\nCreated /kaggle/working/rakshak-ai-lora.zip")

# Also zip the merged model
print("\nPackaging merged model...")
with zipfile.ZipFile(
    "/kaggle/working/rakshak-ai-merged.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for root, dirs, files in os.walk(merged_path):
        for fn in files:
            file_path = os.path.join(root, fn)
            arcname = os.path.relpath(file_path, merged_path)
            zf.write(file_path, arcname)

print("Created /kaggle/working/rakshak-ai-merged.zip")

print("\n" + "=" * 60)
print("Training Complete")
print("=" * 60)
print("\nOutput files:")
print("  1. rakshak-ai-lora.zip — LoRA adapter")
print("  2. rakshak-ai-merged.zip — Merged 16-bit model (SafeTensors)")
print("  3. rakshak-ai.litertlm — LiteRT model for flutter_gemma (if ai-edge-torch succeeded)")
print("\nNext steps:")
print("  1. If .litertlm was exported: transfer rakshak-ai.litertlm to phone")
print("  2. Otherwise: download merged model, run convert_to_litertlm.py locally")
print("  3. Transfer .litertlm to phone, load in Rakshak AI app")